In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

rows, cols = 4, 4
start, goal = (0, 0), (3, 3)
blocked = {(1, 1), (2, 1)}

actions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
q_table = np.zeros((rows * cols, 4))

def state_index(pos):
    return pos[0] * cols + pos[1]

def step(pos, action):
    dr, dc = actions[action]
    nxt = (pos[0] + dr, pos[1] + dc)

    nxt = (
        min(max(nxt[0], 0), rows - 1),
        min(max(nxt[1], 0), cols - 1)
    )

    if nxt in blocked:
        nxt = pos

    done = nxt == goal
    reward = 10 if done else -1

    return nxt, reward, done

print("Grid World Size:", rows, "x", cols)
print("Start State:", start)
print("Goal State:", goal)
print("Blocked States:", blocked)

In [ ]:
alpha, gamma = 0.1, 0.9
epsilon, decay = 1.0, 0.995
rewards = []

for episode in range(2000):
    pos = start
    total_reward = 0

    for _ in range(100):
        s = state_index(pos)

        if np.random.random() < epsilon:
            action = np.random.randint(4)
        else:
            action = np.argmax(q_table[s])

        nxt, reward, done = step(pos, action)
        ns = state_index(nxt)

        target = reward + gamma * np.max(q_table[ns]) * (not done)
        q_table[s, action] += alpha * (target - q_table[s, action])

        pos = nxt
        total_reward += reward

        if done:
            break

    epsilon = max(0.05, epsilon * decay)
    rewards.append(total_reward)

print("Training Episodes:", len(rewards))
print("Final Epsilon:", round(epsilon, 4))
print("Average Reward:", round(np.mean(rewards[-100:]), 2))

In [ ]:
pos = start
path = [pos]
visited = {pos}

for _ in range(100):
    s = state_index(pos)

    valid_actions = [
        a for a, (dr, dc) in enumerate(actions)
        if 0 <= pos[0] + dr < rows
        and 0 <= pos[1] + dc < cols
        and (pos[0] + dr, pos[1] + dc) not in blocked
        and (pos[0] + dr, pos[1] + dc) not in visited
    ]

    if not valid_actions:
        break

    action = max(valid_actions, key=lambda a: q_table[s, a])
    pos, _, done = step(pos, action)

    path.append(pos)
    visited.add(pos)

    if done:
        break

print("Learned Path:", path)
print("Goal Reached:", pos == goal)

plt.figure(figsize=(7, 3))
plt.plot(rewards)
plt.xlabel("Episode")
plt.ylabel("Reward")
plt.title("Q-Learning Performance")
plt.grid(alpha=0.25)
plt.show()

grid = np.zeros((rows, cols))

for r, c in blocked:
    grid[r, c] = -1

grid[goal] = 2

plt.figure(figsize=(5, 5))
plt.imshow(grid, cmap="coolwarm")

for r in range(rows):
    for c in range(cols):
        position = (r, c)
        label = (
            "S" if position == start else
            "G" if position == goal else
            "X" if position in blocked else
            "●" if position in path else ""
        )
        plt.text(c, r, label, ha="center", va="center", fontsize=14)

plt.xticks(range(cols))
plt.yticks(range(rows))
plt.title("Learned Grid World Path")
plt.grid(alpha=0.25)
plt.show()